# 01 — PySR run

This is the only active notebook that may fit PySR. It is intentionally dry-run by default. Set `ALLOW_FIT = True` only under an explicit, reviewed execution contract. This notebook saves continuous scores; AUC calculations belong in `02_auc_analysis.ipynb`.

All results remain provisional, unverified, and pending review.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight

def repository_root() -> Path:
    candidate = Path.cwd().resolve()
    for _ in range(6):
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
        candidate = candidate.parent
    raise FileNotFoundError('Could not locate repository root from notebook directory.')

ROOT = repository_root()
DATA_PATH = ROOT / 'data' / 'raw' / 'masses_exclusions.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'pysr' / 'notebook_pysr_run_v1'
RUN_ID = OUTPUT_DIR.name
FEATURES = ['mchi1', 'mchipm1']
TARGET = 'exclusion'
POSITIVE_LABEL = 1
TEST_SIZE = 0.20
PYTHON_SEED = 42
SPLIT_SEED = 42
PYSR_SEED = 42
BINARY_OPERATORS = ['+', '-', '*']
UNARY_OPERATORS = []
NITERATIONS = 60
POPULATIONS = 2
POPULATION_SIZE = 20
MAXSIZE = 30
PARSIMONY = 0.0001
TIMEOUT_IN_SECONDS = 300
ALLOW_FIT = False

RUN_CONTRACT = {
    'run_id': RUN_ID,
    'dataset_id': 'masses_exclusions',
    'raw_path': str(DATA_PATH.relative_to(ROOT)),
    'features': FEATURES,
    'target': TARGET,
    'positive_label': POSITIVE_LABEL,
    'forbidden_columns': ['Final_CLs'],
    'split': {'method': 'stratified', 'test_size': TEST_SIZE, 'seed': SPLIT_SEED},
    'seeds': {'python': PYTHON_SEED, 'split': SPLIT_SEED, 'pysr': PYSR_SEED},
    'operators': {'binary': BINARY_OPERATORS, 'unary': UNARY_OPERATORS},
    'search': {'niterations': NITERATIONS, 'populations': POPULATIONS, 'population_size': POPULATION_SIZE, 'maxsize': MAXSIZE, 'parsimony': PARSIMONY, 'timeout_in_seconds': TIMEOUT_IN_SECONDS},
    'review_status': 'provisional, unverified, pending review',
}

np.random.seed(PYTHON_SEED)
print(f'Notebook root: {ROOT}')
print(f'Dataset: {DATA_PATH}')
print(f'Output: {OUTPUT_DIR}')
print(f'ALLOW_FIT={ALLOW_FIT}')

In [ ]:
def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return 'not installed'

environment = {
    'python': sys.version.split()[0],
    'packages': {name: package_version(name) for name in ['jupyter', 'ipykernel', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'PyYAML', 'pysr', 'juliacall']},
    'notebook_fit_default': ALLOW_FIT,
}
display(pd.DataFrame([{'package': k, 'version': v} for k, v in environment['packages'].items()]))

if 'Final_CLs' in FEATURES or TARGET == 'Final_CLs':
    raise ValueError('Final_CLs is audit-only and cannot be used as a feature or target.')
if not (0.0 < TEST_SIZE < 1.0):
    raise ValueError('TEST_SIZE must be between 0 and 1.')
if not DATA_PATH.is_file():
    raise FileNotFoundError(DATA_PATH)
if not OUTPUT_DIR.is_relative_to(ROOT / 'outputs' / 'pysr'):
    raise ValueError('Output must remain under outputs/pysr/.')


In [ ]:
data = pd.read_csv(DATA_PATH)
required = FEATURES + [TARGET]
missing = [column for column in required if column not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
if data[required].isna().any().any():
    raise ValueError('Features and target must not contain missing values.')
if set(data[TARGET].unique().tolist()) != {0, 1}:
    raise ValueError(f'Expected binary target values {{0, 1}}, found {sorted(data[TARGET].unique())}')
if any(not pd.api.types.is_numeric_dtype(data[column]) for column in FEATURES):
    raise TypeError('All model features must be numeric.')

train_rows, test_rows = train_test_split(
    np.arange(len(data)),
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=data[TARGET],
)
X_train = data.iloc[train_rows][FEATURES].reset_index(drop=True)
X_test = data.iloc[test_rows][FEATURES].reset_index(drop=True)
y_train = data.iloc[train_rows][TARGET].to_numpy()
y_test = data.iloc[test_rows][TARGET].to_numpy()
sample_weights = compute_sample_weight('balanced', y_train)
print(f'Rows: {len(data)}; train: {len(X_train)}; test: {len(X_test)}')
display(data[FEATURES + [TARGET]].describe())

In [ ]:
model = None
score_frame = None
run_status = 'dry_run'

if not ALLOW_FIT:
    print('DRY RUN: validation and split completed; PySR fit was not executed.')
else:
    if OUTPUT_DIR.exists():
        raise FileExistsError(f'Refusing to overwrite existing output: {OUTPUT_DIR}')
    OUTPUT_DIR.mkdir(parents=True)
    (OUTPUT_DIR / 'pysr_workspace').mkdir()
    from pysr import PySRRegressor

    model = PySRRegressor(
        niterations=NITERATIONS,
        populations=POPULATIONS,
        population_size=POPULATION_SIZE,
        maxsize=MAXSIZE,
        parsimony=PARSIMONY,
        timeout_in_seconds=TIMEOUT_IN_SECONDS,
        binary_operators=BINARY_OPERATORS,
        unary_operators=UNARY_OPERATORS,
        random_state=PYSR_SEED,
        parallelism='serial',
        precision=32,
        deterministic=True,
        warm_start=False,
        model_selection='best',
        elementwise_loss='loss(prediction, target, weight) = weight * (prediction - target)^2',
        temp_equation_file=False,
        delete_tempfiles=False,
        output_directory=str(OUTPUT_DIR / 'pysr_workspace'),
        run_id=RUN_ID,
        progress=True,
    )
    model.fit(X_train, y_train, weights=sample_weights)
    scores = np.asarray(model.predict(X_test), dtype=float).reshape(-1)
    if len(scores) != len(y_test) or not np.isfinite(scores).all():
        raise ValueError('PySR returned invalid continuous test scores.')
    score_frame = pd.DataFrame({
        'row_index': data.iloc[test_rows].index.to_numpy(),
        'split': 'test',
        'y_true': y_test,
        'score': scores,
        'score_source': 'PySRRegressor.predict',
    })
    run_status = 'completed'
    print(f'Completed PySR fit with {len(score_frame)} continuous test scores.')

In [ ]:
if score_frame is None:
    print('No artifacts written in dry-run mode. Set ALLOW_FIT=True only when authorized.')
else:
    score_frame.to_csv(OUTPUT_DIR / 'scores.csv', index=False)
    model.equations_.to_csv(OUTPUT_DIR / 'equations.csv', index=False)
    with (OUTPUT_DIR / 'model.pkl').open('wb') as handle:
        pickle.dump(model, handle)
    environment['fit_status'] = run_status
    environment['created_utc'] = datetime.now(timezone.utc).isoformat()
    (OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n', encoding='utf-8')
    metadata_payload = {**RUN_CONTRACT, 'fit_status': run_status, 'created_utc': environment['created_utc'], 'output_dir': str(OUTPUT_DIR.relative_to(ROOT))}
    (OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata_payload, indent=2) + '\n', encoding='utf-8')
    (OUTPUT_DIR / 'run.log').write_text('PySR notebook run completed. AUC was intentionally not calculated here.\n', encoding='utf-8')
    print(f'Wrote run artifacts to {OUTPUT_DIR}')